MCF ITB 2025 - Code Documentation

In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

In [4]:
data_claim = pd.read_csv("data/Data_Klaim.csv")

In [5]:
data_claim

,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,Nominal Klaim Yang Disetujui,Nominal Biaya RS Yang Terjadi,Lokasi RS
0,C-0001-M,POL-0176,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,2024-07-08,2024-05-27,2024-05-27,2.809365e+07,6.143948e+06,Singapore
1,C-0002-M,POL-3288,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-08-06,2024-07-15,2024-07-15,8.098728e+07,8.230952e+07,Malaysia
2,C-0003-M,POL-1786,R,OP,C18.9,"MALIGNANT NEOPLASM, COLON, UNSPECIFIED",PAID,2024-10-17,2024-05-16,2024-05-16,1.830471e+08,1.928599e+08,Singapore
3,C-0004-M,POL-1786,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-09-03,2024-07-18,2024-07-18,1.914244e+08,1.914244e+08,Singapore
4,C-0005-M,POL-2778,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,NaN,2024-06-06,2024-06-06,1.389364e+08,1.389364e+08,Singapore
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4622,C-5777-M,POL-3288,C,IP,C34,Malignant neoplasm of bronchus and lung,PAID,2025-07-02,2025-03-11,2025-03-11,1.513929e+07,1.531253e+07,Malaysia
4623,C-5778-M,POL-3288,C,IP,C34,Malignant neoplasm of bronchus and lung,PAID,2025-07-08,2025-04-18,2025-04-18,6.336514e+07,6.620405e+07,Malaysia
4624,C-5779-M,POL-3288,C,IP,C34.1,"Malignant neoplasm of upper lobe, bronchus or ...",PAID,NaN,2025-05-21,2025-05-21,1.632886e+07,1.651873e+07,Malaysia
4625,C-5780-M,POL-3288,C,IP,C34.9,"Malignant neoplasm of bronchus or lung, unspec...",PAID,NaN,2025-06-30,2025-06-30,6.540838e+07,6.662646e+07,Malaysia


In [6]:
for feature in data_claim.columns:
    print("{} has {} unique value". format(feature,data_claim[feature].unique()))
    print('-'*20)

Claim ID has ['C-0001-M' 'C-0002-M' 'C-0003-M' ... 'C-5779-M' 'C-5780-M' 'C-5781-M'] unique value
--------------------
Nomor Polis has ['POL-0176' 'POL-3288' 'POL-1786' ... 'POL-2878' 'POL-3156' 'POL-0264'] unique value
--------------------
Reimburse/Cashless has ['R' 'C'] unique value
--------------------
Inpatient/Outpatient has ['OP' 'ODC' 'IP' nan 'ODS'] unique value
--------------------
ICD Diagnosis has ['C50' 'C34' 'C18.9' 'C90.0' 'C82.9' 'L02.2' 'A09' 'A90' 'S00.0' 'H26.9'
 'K29.7' 'K57' 'I49.5' 'R50.9' 'J18' 'M17' 'I62.0' 'I67.1' 'C56' 'H26'
 'M54.5' 'G57.1' 'S82.2' 'I63.9' 'H25.9' 'I95.9' 'L03.1' 'L03.9' 'L12.0'
 'J96' 'H35.9' 'L03' 'I67' 'J70.4' 'J84.9' 'K29' 'R10' 'J20' 'S52' 'S52.0'
 'N18.0' 'N23' 'N18.6' 'I87' 'G62.9' 'D07.5' 'C61' 'N70.1' 'N70' 'S09'
 'I25.1' 'K63.5' nan 'R11' 'G40.2' 'R56.9' 'N20.1' 'N20' 'D25.9' 'E86.0'
 'R57' 'N39.0' 'R07.4' 'K63.8' 'M75.1' 'A49' 'I84' 'K30' 'H40.2' 'C85.9'
 'H35' 'H35.0' 'K63.9' 'K21' 'I63.6' 'I63' 'M48.0' 'K80.0' 'H25.1' 'M67.4'
 'H

In [7]:
data_claim.isna().sum()/data_claim.shape[0]*100 #persentase null per feature nya, klo misal lebih dari 30% apus aja columnya

Claim ID                         0.000000
Nomor Polis                      0.000000
Reimburse/Cashless               0.000000
Inpatient/Outpatient             0.799654
ICD Diagnosis                    0.129674
ICD Description                  0.129674
Status Klaim                     0.000000
Tanggal Pembayaran Klaim         0.799654
Tanggal Pasien Masuk RS          0.000000
Tanggal Pasien Keluar RS         0.000000
Nominal Klaim Yang Disetujui     0.000000
Nominal Biaya RS Yang Terjadi    0.000000
Lokasi RS                        0.151286
dtype: float64

In [8]:
data_claim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4627 entries, 0 to 4626
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Claim ID                       4627 non-null   object 
 1   Nomor Polis                    4627 non-null   object 
 2   Reimburse/Cashless             4627 non-null   object 
 3   Inpatient/Outpatient           4590 non-null   object 
 4   ICD Diagnosis                  4621 non-null   object 
 5   ICD Description                4621 non-null   object 
 6   Status Klaim                   4627 non-null   object 
 7   Tanggal Pembayaran Klaim       4590 non-null   object 
 8   Tanggal Pasien Masuk RS        4627 non-null   object 
 9   Tanggal Pasien Keluar RS       4627 non-null   object 
 10  Nominal Klaim Yang Disetujui   4627 non-null   float64
 11  Nominal Biaya RS Yang Terjadi  4627 non-null   float64
 12  Lokasi RS                      4620 non-null   o

In [9]:
# pastikan datetime
data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(
    data_claim['Tanggal Pasien Masuk RS']
)

# buat kolom tahun dan bulan dari tanggal masuk
data_claim['Tahun'] = data_claim['Tanggal Pasien Masuk RS'].dt.year
data_claim['Bulan'] = data_claim['Tanggal Pasien Masuk RS'].dt.month

# ambil minimum per claim dulu
data_claim['Claim_Dibayar'] = np.minimum(
    data_claim['Nominal Biaya RS Yang Terjadi'],
    data_claim['Nominal Klaim Yang Disetujui']
)

# group per bulan berdasarkan tanggal masuk
monthly = data_claim.groupby(['Tahun','Bulan']).agg(
    Number_of_Claims = ('Claim ID','count'),
    Total_Claim_Bulanan = ('Claim_Dibayar','sum'),
    Unique_Polis = ('Nomor Polis','nunique')
).reset_index()


# hasil akhir
final = monthly[['Tahun','Bulan','Number_of_Claims','Total_Claim_Bulanan', 'Unique_Polis']]

final.head()


,Tahun,Bulan,Number_of_Claims,Total_Claim_Bulanan,Unique_Polis
0,2024,1,302,2.012259e+10,179
1,2024,2,208,1.385533e+10,120
2,2024,3,278,1.430939e+10,174
3,2024,4,239,1.143551e+10,138
4,2024,5,263,1.216403e+10,152


In [10]:
final

,Tahun,Bulan,Number_of_Claims,Total_Claim_Bulanan,Unique_Polis
0,2024,1,302,2.012259e+10,179
1,2024,2,208,1.385533e+10,120
2,2024,3,278,1.430939e+10,174
3,2024,4,239,1.143551e+10,138
4,2024,5,263,1.216403e+10,152
5,2024,6,225,1.187225e+10,130
6,2024,7,257,1.492223e+10,142
7,2024,8,228,1.351294e+10,138
8,2024,9,208,1.226412e+10,121
9,2024,10,274,1.268117e+10,158


XG BOOST 

In [11]:
# pastikan urut waktu
final = final.sort_values(['Tahun','Bulan']).reset_index(drop=True)

final['date'] = pd.to_datetime(final['Tahun'].astype(str) + '-' + final['Bulan'].astype(str))
final = final.set_index('date')

targets = [
    'Number_of_Claims',
    'Total_Claim_Bulanan',
    'Unique_Polis'
]

In [12]:
def create_lags(data, target, n_lags):
    df = data[[target]].copy()
    for i in range(1, n_lags+1):
        df[f'lag_{i}'] = df[target].shift(i)
    return df.dropna()


In [13]:
def train_and_forecast(ts, target):

    tscv = TimeSeriesSplit(n_splits=3)
    results = {}
    
    # cari lag terbaik
    for lag in range(1,7):
        df_lag = create_lags(ts, target, lag)
        
        X = df_lag.drop(columns=[target])
        y = df_lag[target]
        
        rmse_list = []
        
        for train_idx, test_idx in tscv.split(X):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            
            model = XGBRegressor(
                n_estimators=200,
                max_depth=3,
                learning_rate=0.05,
                random_state=42
            )
            
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            rmse = np.sqrt(mean_squared_error(y_test, preds))
            rmse_list.append(rmse)
        
        results[lag] = np.mean(rmse_list)
    
    best_lag = min(results, key=results.get)
    
    # train final model
    df_lag = create_lags(ts, target, best_lag)
    X = df_lag.drop(columns=[target])
    y = df_lag[target]
    
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        random_state=42
    )
    
    model.fit(X, y)
    
    # forecasting 5 bulan ke depan
    future_preds = []
    last_data = ts[[target]].copy()
    
    for i in range(5):
        df_temp = create_lags(last_data, target, best_lag)
        last_row = df_temp.iloc[-1:]
        
        X_pred = last_row.drop(columns=[target])
        pred = model.predict(X_pred)[0]
        
        # bulatkan untuk count data
        if target in ['Number_of_Claims','Unique_Polis']:
            pred = round(pred)
        
        future_preds.append(pred)
        
        next_month = last_data.index[-1] + pd.DateOffset(months=1)
        last_data.loc[next_month] = pred
    
    return best_lag, future_preds


In [14]:
forecast_dict = {}
lag_dict = {}

for target in targets:
    best_lag, preds = train_and_forecast(final, target)
    forecast_dict[target] = preds
    lag_dict[target] = best_lag

print("Best Lag per Target:", lag_dict)

future_dates = pd.date_range(start='2025-08-01', periods=5, freq='MS')

forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Predicted_Number_of_Claims': forecast_dict['Number_of_Claims'],
    'Predicted_Total_Claim': forecast_dict['Total_Claim_Bulanan'],
    'Predicted_Unique_Polis': forecast_dict['Unique_Polis']
})

forecast_df


Best Lag per Target: {'Number_of_Claims': 5, 'Total_Claim_Bulanan': 6, 'Unique_Polis': 6}


,Date,Predicted_Number_of_Claims,Predicted_Total_Claim,Predicted_Unique_Polis
0,2025-08-01,264,1.369914e+10,147
1,2025-09-01,238,1.057042e+10,146
2,2025-10-01,231,1.219987e+10,137
3,2025-11-01,216,1.410730e+10,139
4,2025-12-01,242,1.343775e+10,143


In [15]:
forecast_df['Predicted_Claim_Frequency'] = (
    forecast_df['Predicted_Number_of_Claims'] /
    forecast_df['Predicted_Unique_Polis']
) * 100

forecast_df['Predicted_Claim_Severity'] = (
    forecast_df['Predicted_Total_Claim'] /
    forecast_df['Predicted_Number_of_Claims']
)


In [16]:
forecast_df

,Date,Predicted_Number_of_Claims,Predicted_Total_Claim,Predicted_Unique_Polis,Predicted_Claim_Frequency,Predicted_Claim_Severity
0,2025-08-01,264,1.369914e+10,147,179.591837,5.189067e+07
1,2025-09-01,238,1.057042e+10,146,163.013699,4.441352e+07
2,2025-10-01,231,1.219987e+10,137,168.613139,5.281329e+07
3,2025-11-01,216,1.410730e+10,139,155.395683,6.531158e+07
4,2025-12-01,242,1.343775e+10,143,169.230769,5.552789e+07


**Conclusion : dapet MAPE : 15.59937**


Random Forest

In [17]:
from sklearn.ensemble import RandomForestRegressor

In [18]:
def train_and_forecast_rf(ts, target):

    tscv = TimeSeriesSplit(n_splits=3)
    results = {}
    
    # cari lag terbaik
    for lag in range(1,7):
        df_lag = create_lags(ts, target, lag)
        
        X = df_lag.drop(columns=[target])
        y = df_lag[target]
        
        rmse_list = []
        
        for train_idx, test_idx in tscv.split(X):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            
            model = RandomForestRegressor(
                n_estimators=300,
                max_depth=5,
                random_state=42
            )
            
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            rmse = np.sqrt(mean_squared_error(y_test, preds))
            rmse_list.append(rmse)
        
        results[lag] = np.mean(rmse_list)
    
    best_lag = min(results, key=results.get)
    
    # train final model
    df_lag = create_lags(ts, target, best_lag)
    X = df_lag.drop(columns=[target])
    y = df_lag[target]
    
    model = RandomForestRegressor(
        n_estimators=400,
        max_depth=5,
        random_state=42
    )
    
    model.fit(X, y)
    
    # forecasting 5 bulan ke depan
    future_preds = []
    last_data = ts[[target]].copy()
    
    for i in range(5):
        df_temp = create_lags(last_data, target, best_lag)
        last_row = df_temp.iloc[-1:]
        
        X_pred = last_row.drop(columns=[target])
        pred = model.predict(X_pred)[0]
        
        if target in ['Number_of_Claims','Unique_Polis']:
            pred = round(pred)
        
        future_preds.append(pred)
        
        next_month = last_data.index[-1] + pd.DateOffset(months=1)
        last_data.loc[next_month] = pred
    
    return best_lag, future_preds

In [19]:
forecast_dict = {}
lag_dict = {}

for target in targets:
    best_lag, preds = train_and_forecast_rf(final, target)
    forecast_dict[target] = preds
    lag_dict[target] = best_lag

print("Best Lag per Target:", lag_dict)

future_dates = pd.date_range(start='2025-08-01', periods=5, freq='MS')

forecast_df_rf = pd.DataFrame({
    'Date': future_dates,
    'Predicted_Number_of_Claims': forecast_dict['Number_of_Claims'],
    'Predicted_Total_Claim': forecast_dict['Total_Claim_Bulanan'],
    'Predicted_Unique_Polis': forecast_dict['Unique_Polis']
})

forecast_df_rf


Best Lag per Target: {'Number_of_Claims': 4, 'Total_Claim_Bulanan': 2, 'Unique_Polis': 4}


,Date,Predicted_Number_of_Claims,Predicted_Total_Claim,Predicted_Unique_Polis
0,2025-08-01,247,1.362824e+10,141
1,2025-09-01,232,1.202455e+10,138
2,2025-10-01,227,1.164754e+10,131
3,2025-11-01,222,1.122594e+10,136
4,2025-12-01,234,1.406110e+10,137


In [20]:
forecast_df_rf['Predicted_Claim_Frequency'] = (
    forecast_df_rf['Predicted_Number_of_Claims'] /
    forecast_df_rf['Predicted_Unique_Polis']
) * 100

forecast_df_rf['Predicted_Claim_Severity'] = (
    forecast_df_rf['Predicted_Total_Claim'] /
    forecast_df_rf['Predicted_Number_of_Claims']
)


In [21]:
forecast_df_rf

,Date,Predicted_Number_of_Claims,Predicted_Total_Claim,Predicted_Unique_Polis,Predicted_Claim_Frequency,Predicted_Claim_Severity
0,2025-08-01,247,1.362824e+10,141,175.177305,5.517505e+07
1,2025-09-01,232,1.202455e+10,138,168.115942,5.182997e+07
2,2025-10-01,227,1.164754e+10,131,173.282443,5.131074e+07
3,2025-11-01,222,1.122594e+10,136,163.235294,5.056731e+07
4,2025-12-01,234,1.406110e+10,137,170.802920,6.009017e+07


MAPE : 21.17863

SARIMAX

In [22]:
import warnings
warnings.filterwarnings("ignore")
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error

In [23]:
def train_sarima(ts, target):

    y = ts[target].asfreq('MS')  # monthly start frequency
    
    # parameter kecil dulu biar cepat
    p = d = q = range(0, 2)
    P = D = Q = range(0, 2)
    s = 12  # seasonality bulanan
    
    best_aic = np.inf
    best_order = None
    best_seasonal = None
    
    for i in p:
        for j in d:
            for k in q:
                for sp in P:
                    for sd in D:
                        for sq in Q:
                            try:
                                model = SARIMAX(
                                    y,
                                    order=(i,j,k),
                                    seasonal_order=(sp,sd,sq,s),
                                    enforce_stationarity=False,
                                    enforce_invertibility=False
                                )
                                results = model.fit(disp=False)
                                
                                if results.aic < best_aic:
                                    best_aic = results.aic
                                    best_order = (i,j,k)
                                    best_seasonal = (sp,sd,sq,s)
                            except:
                                continue

    print(f"Best Order for {target}: {best_order}")
    print(f"Best Seasonal Order: {best_seasonal}")
    
    # Train final model
    model = SARIMAX(
        y,
        order=best_order,
        seasonal_order=best_seasonal,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    
    results = model.fit(disp=False)
    
    # Forecast 5 bulan
    forecast = results.forecast(steps=5)
    
    # bulatkan untuk count
    if target in ['Number_of_Claims','Unique_Polis']:
        forecast = forecast.round()
    
    return forecast


In [24]:
forecast_sarima = {}

for target in targets:
    preds = train_sarima(final, target)
    forecast_sarima[target] = preds.values

future_dates = pd.date_range(start=final.index[-1] + pd.DateOffset(months=1),
                             periods=5, freq='MS')

forecast_df_sarima = pd.DataFrame({
    'Date': future_dates,
    'Predicted_Number_of_Claims': forecast_sarima['Number_of_Claims'],
    'Predicted_Total_Claim': forecast_sarima['Total_Claim_Bulanan'],
    'Predicted_Unique_Polis': forecast_sarima['Unique_Polis']
})

forecast_df_sarima


c:\Users\hi\anaconda3\envs\data_science\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Best Order for Number_of_Claims: (0, 0, 0)
Best Seasonal Order: (0, 1, 1, 12)


c:\Users\hi\anaconda3\envs\data_science\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hi\anaconda3\envs\data_science\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hi\anaconda3\envs\data_science\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Best Order for Total_Claim_Bulanan: (0, 0, 0)
Best Seasonal Order: (0, 1, 1, 12)


c:\Users\hi\anaconda3\envs\data_science\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Best Order for Unique_Polis: (0, 0, 0)
Best Seasonal Order: (0, 1, 1, 12)


,Date,Predicted_Number_of_Claims,Predicted_Total_Claim,Predicted_Unique_Polis
0,2025-08-01,228.0,1.351294e+10,138.0
1,2025-09-01,208.0,1.226412e+10,121.0
2,2025-10-01,274.0,1.268117e+10,158.0
3,2025-11-01,270.0,1.373306e+10,147.0
4,2025-12-01,238.0,1.201391e+10,133.0


In [25]:
forecast_df_sarima['Predicted_Claim_Frequency'] = (
    forecast_df_sarima['Predicted_Number_of_Claims'] /
    forecast_df_sarima['Predicted_Unique_Polis']
) * 100

forecast_df_sarima['Predicted_Claim_Severity'] = (
    forecast_df_sarima['Predicted_Total_Claim'] /
    forecast_df_sarima['Predicted_Number_of_Claims']
)


In [26]:
forecast_df_sarima

,Date,Predicted_Number_of_Claims,Predicted_Total_Claim,Predicted_Unique_Polis,Predicted_Claim_Frequency,Predicted_Claim_Severity
0,2025-08-01,228.0,1.351294e+10,138.0,165.217391,5.926726e+07
1,2025-09-01,208.0,1.226412e+10,121.0,171.900826,5.896211e+07
2,2025-10-01,274.0,1.268117e+10,158.0,173.417722,4.628163e+07
3,2025-11-01,270.0,1.373306e+10,147.0,183.673469,5.086318e+07
4,2025-12-01,238.0,1.201391e+10,133.0,178.947368,5.047861e+07


In [27]:
# Make sure Date column is datetime
forecast_df_sarima["Date"] = pd.to_datetime(forecast_df_sarima["Date"])

submission_rows = []

for _, row in forecast_df_sarima.iterrows():
    
    year = row["Date"].year
    month = f"{row['Date'].month:02d}"
    
    submission_rows.append(
        (f"{year}_{month}_Claim_Frequency", row["Predicted_Claim_Frequency"])
    )
    submission_rows.append(
        (f"{year}_{month}_Claim_Severity", row["Predicted_Claim_Severity"])
    )
    submission_rows.append(
        (f"{year}_{month}_Total_Claim", row["Predicted_Total_Claim"])
    )

submission_df = pd.DataFrame(submission_rows, columns=["id", "value"])

# Optional: remove scientific notation for severity/total
submission_df["value"] = submission_df["value"].astype(float)

# Export to CSV
submission_df.to_csv("submission_sarima.csv", index=False)

print("submission_sarima.csv created successfully")

submission_sarima.csv created successfully
